# Université Chouaïb Doukkali - El Jadida
## Faculté Polydisciplinaire - Sidi Bennour
### Filières : DL - BIBD - CSCN - GA | Semestre 5
### Module : Intelligence Artificielle
---
# Mini-Projet : CardioGuard — Prédiction des Maladies Cardiaques par Machine Learning
**Auteur :** Projet Étudiant  
**Enseignant :** Pr. Abderrahmane Ed-daoudy  

### Objectifs du Mini-Projet :
1. **Collecte & Exploration des Données (EDA)** : Dataset UCI Heart Disease (303 patients, 13 variables cliniques).
2. **Développement de 4 Modèles ML** : Régression Logistique, Arbre de Décision, Random Forest, Support Vector Machine (SVM).
3. **Comparaison & Sélection du Meilleur Modèle** : Évaluation via Accuracy, Précision, Rappel, F1-Score, Courbes ROC-AUC et Matrices de Confusion.
4. **Sérialisation du Modèle** : Sauvegarde sous format `.pkl` (`joblib`).
5. **Déploiement Web** : Intégration dans une interface moderne (React & Streamlit).

## 1. Importation des bibliothèques nécessaires

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

print("Bibliothèques importées avec succès !")

## 2. Chargement et Exploration des Données (EDA)

In [2]:
# Chargement du jeu de données UCI Heart Disease
df = pd.read_csv("ml/data/heart_disease.csv")
print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

In [3]:
# Informations sur les types et valeurs manquantes
df.info()
print("\nValeurs manquantes par colonne :\n", df.isnull().sum())

In [4]:
# Statistiques descriptives
df.describe()

## 3. Préparation des Données (Split & Standardisation)

In [5]:
# Séparation des entrées (X) et de la sortie (y)
X = df.drop("target", axis=1)
y = df["target"]

# Découpage 80% apprentissage (train) et 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Normalisation avec StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Sauvegarde du scaler
joblib.dump(scaler, "ml/models/scaler.pkl")
print("Dimensions train :", X_train.shape, "| Dimensions test :", X_test.shape)

## 4. Entraînement et Évaluation des 4 Modèles ML

In [6]:
models = {
    "Régression Logistique": LogisticRegression(max_iter=1000, random_state=42),
    "Arbre de Décision": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Support Vector Machine": SVC(kernel="rbf", probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        "Accuracy": acc * 100,
        "Précision": prec * 100,
        "Rappel": rec * 100,
        "F1-Score": f1 * 100,
        "ROC-AUC": auc * 100
    }

comparison_df = pd.DataFrame(results).T.round(2)
comparison_df

## 5. Visualisation des Performances (Courbes ROC & Matrices de Confusion)

In [7]:
# Tracé comparatif des Courbes ROC
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.2f})")

plt.plot([0, 1], [0, 1], "k--", label="Aléatoire (AUC = 0.50)")
plt.xlabel("Taux de Faux Positifs (FPR)")
plt.ylabel("Taux de Vrais Positifs (TPR)")
plt.title("Comparaison des Courbes ROC des 4 Modèles")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Choix et Déploiement du Meilleur Modèle (.pkl)

In [8]:
# Sélection automatique selon le meilleur F1-Score
best_name = max(results, key=lambda k: results[k]["F1-Score"])
best_model = models[best_name]
print(f"Modèle Retenu : {best_name}")

# Exportation du modèle au format .pkl avec joblib
joblib.dump(best_model, "ml/models/best_model.pkl")
print("Modèle exporté avec succès sous : ml/models/best_model.pkl")

## 7. Test de Prédiction sur un Nouveau Patient (Inférence)

In [9]:
# Exemple de test : Homme de 63 ans avec caractéristiques cliniques à risque
nouveau_patient = np.array([[63, 1, 3, 145, 233, 1, 0, 150, 0, 2.3, 0, 0, 1]])
nouveau_patient_scaled = scaler.transform(nouveau_patient)

prediction = best_model.predict(nouveau_patient_scaled)[0]
probabilite = best_model.predict_proba(nouveau_patient_scaled)[0][1]

print("=== RÉSULTAT DU DIAGNOSTIC ===")
print("Diagnostic :", "Cardiopathie Détectée (1)" if prediction == 1 else "Patient Sain (0)")
print(f"Probabilité estimée : {probabilite*100:.2f}%")